In [2]:
# Pipeline de Nettoyage de Données

import pandas as pd
import math

# ================================
# 1 - DONNÉES BRUTES
# ================================

data = {
    "nom":     ["Alice", "Bob", None, "David", "Emma", "Frank"],
    "age":     [25, None, 35, 28, None, 42],
    "ville":   ["Paris", "Lyon", "Paris", None, "Lyon", "Marseille"],
    "salaire": [3000, 4500, None, 2800, 5200, None],
    "genre":   ["F", "M", "M", "M", "F", "M"]
}

df = pd.DataFrame(data)

print("DONNEES ORIGINALES")
print(df)
print(f"\nValeurs manquantes :\n{df.isnull().sum()}")

# ================================
# 2 - VALEURS MANQUANTES
# ================================

# Supprimer les lignes sans nom (colonne essentielle)
df = df.dropna(subset=["nom"])

# Remplacer les ages manquants par la moyenne
moyenne_age     = df["age"].mean()
df["age"]       = df["age"].fillna(moyenne_age)

# Remplacer les salaires manquants par la médiane
df_tries        = df["salaire"].dropna().sort_values().tolist()
n               = len(df_tries)
mediane_salaire = (df_tries[n//2 - 1] + df_tries[n//2]) / 2
df["salaire"]   = df["salaire"].fillna(mediane_salaire)

# Remplacer les villes manquantes par "Inconnu"
df["ville"]     = df["ville"].fillna("Inconnu")

print("\n\nAPRES TRAITEMENT VALEURS MANQUANTES")
print(df)

# ================================
# 3 - ENCODAGE
# ================================

# Encodage du genre : F → 0, M → 1
df["genre"] = df["genre"].map({"F": 0, "M": 1})

# One-hot encoding de la ville (une colonne par ville)
df = pd.get_dummies(df, columns=["ville"])

print("\n\nAPRES ENCODAGE")
print(df)

# ================================
# 4 - NORMALISATION
# ================================

# Min-Max : ramène les valeurs entre 0 et 1
#
#          valeur - min
# formule: ────────────
#          max  - min

for colonne in ["age", "salaire"]:
    mini        = df[colonne].min()
    maxi        = df[colonne].max()
    df[colonne] = (df[colonne] - mini) / (maxi - mini)
    df[colonne] = df[colonne].round(2)

print("\n\nAPRES NORMALISATION (valeurs entre 0 et 1)")
print(df)

# ================================
# 5 - SAUVEGARDE
# ================================

df.to_csv("donnees_nettoyees.csv", index=False)
print("\nFichier sauvegarde : donnees_nettoyees.csv")

DONNEES ORIGINALES
     nom   age      ville  salaire genre
0  Alice  25.0      Paris   3000.0     F
1    Bob   NaN       Lyon   4500.0     M
2    NaN  35.0      Paris      NaN     M
3  David  28.0        NaN   2800.0     M
4   Emma   NaN       Lyon   5200.0     F
5  Frank  42.0  Marseille      NaN     M

Valeurs manquantes :
nom        1
age        2
ville      1
salaire    2
genre      0
dtype: int64


APRES TRAITEMENT VALEURS MANQUANTES
     nom        age      ville  salaire genre
0  Alice  25.000000      Paris   3000.0     F
1    Bob  31.666667       Lyon   4500.0     M
3  David  28.000000    Inconnu   2800.0     M
4   Emma  31.666667       Lyon   5200.0     F
5  Frank  42.000000  Marseille   3750.0     M


APRES ENCODAGE
     nom        age  salaire  genre  ville_Inconnu  ville_Lyon  \
0  Alice  25.000000   3000.0      0          False       False   
1    Bob  31.666667   4500.0      1          False        True   
3  David  28.000000   2800.0      1           True       False   